# Process plate counts to get ratios of variants in initial pool

An initial pooled library was made by adding equal volumes of all variants and then infecting this pool on MDCK-SIAT1 cells. These infections were done by serially diluting a starting 50 uL volume of pool. Barcodes were then isolated and sequenced so that we can determine representation of each variant in the pool, as well as the appropriate library pool dilution (i.e., MOI) to use in neutralization assays. 

The plots generated by this notebook are interactive, so you can mouseover points for details, use the mouse-scroll to zoom and pan, and use interactive dropdowns at the bottom of the plots.

## Setup
Import Python modules:

In [1]:
import pickle
import sys

import altair as alt

import numpy
import string
import pandas as pd
from os.path import join
import os
import ruamel.yaml as yaml

_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
               ]

In [2]:
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok=True)

## Add input data locations
Some of these files are defined as data, and some of these files are generated by running the specified library pooling data as `miscellaneous_plates` through the `seqneut-pipeline`. For details on how these files are generated, see the `README.md' in [https://github.com/jbloomlab/seqneut-pipeline](https://github.com/jbloomlab/seqneut-pipeline)

In [3]:
# Viral library contents and barcode IDs
viral_library_csv =  '../../../data/viral_libraries/flu-seqneut-2025to2026-barcode-to-strain-designed.csv'
# Neutralization standard set of barcode IDs
neut_standard_set_csv =  '../../../data/neut_standard_sets/loes2023_neut_standards.csv'
# All samples included in library poolign sequencing run
# Contains information on library, dilution factor, R1 location
samplesfile = '../../../data/miscellaneous_plates/2026-01-08_equal_vol_pool.csv'

## CHANGE THIS EACH TIME YOU RUN NB ##
# Counts and fates files output by running library pooling samples as miscellaneous plates
platedir = '../../../results/miscellaneous_plates/20260108_equal_vol_pool/'

# Identify all counts and fates CSVs
count_csvs = []
fate_csvs = []
file_list = os.listdir(platedir)
for f in file_list:
    location = platedir + f
    if "_counts" in f:
        count_csvs.append(location)
    elif "_fates" in f:
        fate_csvs.append(location)

In [4]:
# Define a samples dataframe using the samples file
samples_df = pd.read_csv(samplesfile)
samples_df.drop(columns=['fastq'], inplace=True)
samples_df['sample'] = samples_df.apply(
    lambda x: '-'.join(x.astype(str)), axis=1
)

samples = samples_df["sample"].unique().tolist()
print(f"There are {len(samples)} barcode runs.")

samples_df

There are 16 barcode runs.


,well,serum,dilution_factor,replicate,sample
0,A1,none,4,1,A1-none-4-1
1,B1,none,8,1,B1-none-8-1
2,C1,none,16,1,C1-none-16-1
3,D1,none,32,1,D1-none-32-1
4,E1,none,64,1,E1-none-64-1
5,F1,none,128,1,F1-none-128-1
6,G1,none,256,1,G1-none-256-1
7,H1,none,512,1,H1-none-512-1
8,A2,none,4,2,A2-none-4-2
9,B2,none,8,2,B2-none-8-2


## Statistics on barcode-parsing for each sample
Make interactive chart of the "fates" of the sequencing reads parsed for each sample on the plate.

If most sequencing reads are not "valid barcodes", this could potentially indicate some problem in the sequencing or barcode set you are parsing.

Potential fates are:
 - *valid barcode*: barcode that matches a known virus or neutralization standard, we hope most reads are this.
 - *invalid barcode*: a barcode with proper flanking sequences, but does not match a known virus or neutralization standard. If you  have a lot of reads of this type, it is probably a good idea to look at the invalid barcode CSVs (in the `./results/barcode_invalid/` subdirectory created by the pipeline) to see what these invalid barcodes are.
 - *unparseable barcode*: could not parse a barcode from this read as there was not a sequence of the correct length with the appropriate flanking sequence.
 - *low quality barcode*: low-quality or `N` nucleotides in barcode, could indicate problem with sequencing.
 - *failed chastity filter*: reads that failed the Illumina chastity filter, if these are reported in the FASTQ (they may not be).

Also, if the number of reads per sample is very uneven, that could indicate that you did not do a good job of balancing the different samples in the Illumina sequencing.

In [5]:
fates = (
    pd.concat([pd.read_csv(f).assign(well=f.strip(platedir).strip('_fates.csv')) for f, s in zip(fate_csvs, samples)])
    .merge(samples_df, validate="many_to_one", on="well")
    .assign(
        fate_counts=lambda x: x.groupby("fate")["count"].transform("sum"),
        sample_well=lambda x: x["sample"] + " (" + x["well"] + ")",
    )
    .query("fate_counts > 0")[  # only keep fates with at least one count
        ["fate", "count", "well", "sample_well", "dilution_factor"]
    ]
)

assert len(fates) == len(fates.drop_duplicates())


sample_wells = list(
    fates.sort_values(["dilution_factor"])["sample_well"]
)



fates_chart = (
    alt.Chart(fates)
    .encode(
        alt.X("count", scale=alt.Scale(nice=False, padding=3)),
        alt.Y(
            "sample_well",
            title=None,
            sort=sample_wells,
        ),
        alt.Color("fate", sort=sorted(fates["fate"].unique(), reverse=True)),
        alt.Order("fate", sort="descending"),
        tooltip=fates.columns.tolist(),
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=200,
        title=f"Barcode parsing for initial titering plate",
    )
    .configure_axis(grid=False)
)

fates_chart

alt.Chart(...)

## Read barcode counts
Read the counts per barcode:

In [6]:
# get barcode counts
counts = (
    pd.concat([pd.read_csv(c).assign(well=c.replace(platedir,'').strip('_counts.csv')) for c, s in zip(count_csvs, samples)])
    .merge(samples_df, validate="many_to_one", on="well")
    .drop(columns=["replicate"])
    .assign(sample_well=lambda x: x["sample"] + " (" + x["well"] + ")")
)

# classify barcodes as viral or neut standard
barcode_class = pd.concat(
    [
        pd.read_csv(viral_library_csv)[["barcode", "strain"]].assign(
            neut_standard=False,
        ),
        pd.read_csv(neut_standard_set_csv)[["barcode"]].assign(
            neut_standard=True,
            strain=pd.NA,
        ),
    ],
    ignore_index=True,
)

counts

# merge counts and classification of barcodes
assert set(counts["barcode"]) == set(barcode_class["barcode"])
counts = counts.merge(barcode_class, on="barcode", validate="many_to_one")
assert set(sample_wells) == set(counts["sample_well"])

In [7]:
counts

,barcode,count,well,serum,dilution_factor,sample,sample_well,strain,neut_standard
0,CAAATATAATGTCCTG,142645,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),A/SouthAustralia/2525607496/2025_H3N2,False
1,CGGGTCCTAGGCGAGG,108610,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),A/Bangkok/P2391/2025_H3N2,False
2,GTACAAACCTGCAAAT,75688,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
3,GTATAGAAATGGGTCA,69002,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),A/HongKong/1257/2025_H3N2,False
4,GACTTCTTCCGCTCTT,68275,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),A/Michigan/110/2025_H3N2,False
...,...,...,...,...,...,...,...,...,...
3899,TAGACTGGATACGTGA,0,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/Catalonia/NSAV198331894/2025_H1N1,False
3900,TCCGCACTGCGATCAC,0,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/DistrictOfColumbia/27/2023_H3N2,False
3901,TCTTCAAGTCGTGTTA,0,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/England/1845724/2025_H3N2,False
3902,TGGACTGGCCATCCTA,0,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/SuratThani/P3371/2025_H3N2,False


## Average counts per barcode in each well

Plot average counts per barcode.
If a sample has inadequate barcode counts, it may not have good enough statistics for accurate analysis, and a QC-threshold is applied:

In [8]:
avg_barcode_counts = (
    counts.groupby(
        ["well", "sample_well"],
        dropna=False,
        as_index=False,
    )
    .aggregate(avg_count=pd.NamedAgg("count", "mean"))
    .assign(
        fails_qc=lambda x: (
            x["avg_count"] < 500
        ),
    )
)

avg_barcode_counts_chart = (
    alt.Chart(avg_barcode_counts)
    .encode(
        alt.X(
            "avg_count",
            title="average barcode counts per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min barcode count threshold'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if avg_barcode_counts[c].dtype == float else c
            for c in avg_barcode_counts.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Average barcode counts per well for titering plate",
    )
    .configure_axis(grid=False)
)

display(avg_barcode_counts_chart)

# drop wells failing QC
avg_barcode_counts_per_well_drops = list(avg_barcode_counts.query("fails_qc")["well"])

alt.Chart(...)

## Fraction of counts from neutralization standard
Determine the fraction of counts from the neutralization standard in each sample, and make sure this fraction passess the QC threshold.

In [9]:
neut_standard_fracs = (
    counts.assign(
        neut_standard_count=lambda x: x["count"] * x["neut_standard"].astype(int)
    )
    .groupby(
        ["well", "sample_well", 'dilution_factor'],
        dropna=False,
        as_index=False,
    )
    .aggregate(
        total_count=pd.NamedAgg("count", "sum"),
        neut_standard_count=pd.NamedAgg("neut_standard_count", "sum"),
    )
    .assign(
        neut_standard_frac=lambda x: x["neut_standard_count"] / x["total_count"],
        fails_qc=lambda x: (
            x["neut_standard_frac"] < 0.001
        ),
    )
)

neut_standard_fracs_chart = (
    alt.Chart(neut_standard_fracs)
    .encode(
        alt.X(
            "neut_standard_frac",
            title="frac counts from neutralization standard per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min_neut_standard_frac_per_well'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if neut_standard_fracs[c].dtype == float else c
            for c in neut_standard_fracs.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Neutralization-standard fracs per well for titering plate, initial pool",
    )
    .configure_axis(grid=False)
    .configure_legend(titleLimit=1000)
)

display(neut_standard_fracs_chart)

# drop wells failing QC
min_neut_standard_frac_per_well_drops = list(
    neut_standard_fracs.query("fails_qc")["well"]
)

alt.Chart(...)

In [10]:
# Scatterplot of the same data as above, plotted by dilution factor
alt.Chart(neut_standard_fracs).mark_circle(size=60).encode(
    alt.X('dilution_factor:Q', 
          scale=alt.Scale(type='log'),
          title='library pool reciprocal dilution factor'),
    alt.Y('neut_standard_frac:Q', 
          scale=alt.Scale(type='log'),
          title='fraction of reads = neutralization standard'),
    color='fails_qc',
    tooltip=['well', 'dilution_factor', 'neut_standard_frac', 'total_count']
).interactive()

alt.Chart(...)

## Rebalancing strains contained in the library
Viruses were rescued and blind passaged individually. To make the initial pool, we added equal volumes of all strains together and re-infected MDCK-SIAT1 cells. Now we can assess the contribution of each strain to the pool, and determine how much should be added of each virus to achieve more equal balancing. 

Each of the 3 viral barcodes associated with each strain were pooled prior to rescue, so they cannot be balanced. 

In [11]:
# Get summed barcode counts for all strains across all wells
straincounts_allbarcodes = (counts.groupby(['sample','sample_well','strain','dilution_factor','serum','well'])
                          .sum()
                          .reset_index()
                          .drop(columns = ['sample_well', 'neut_standard', 'barcode'])
                         )

# Get sum of all virus/barcode counts per well
sumperwell = (straincounts_allbarcodes.groupby(['sample','dilution_factor','serum','well'])
              .sum()
              .drop(columns=['strain'])
              .reset_index()
              .rename(columns={'count':'counts_perwell'})
             )

# Merge dataframes and calculate fraction of each well devoted to each strain
merged_df = straincounts_allbarcodes.merge(sumperwell, on=['sample','dilution_factor','serum','well'])
merged_df['fraction_strain'] = merged_df['count'] /merged_df['counts_perwell'] / 2
merged_df

,sample,strain,dilution_factor,serum,well,count,counts_perwell,fraction_strain
0,A1-none-4-1,A/Aberystwyth/5654/2025_H3N2,4,none,A1,0,4657768,0.000000
1,A1-none-4-1,A/Alagoas/3579/2025_H1N1,4,none,A1,3267,4657768,0.000351
2,A1-none-4-1,A/Alagoas/4130/2025_H3N2,4,none,A1,120127,4657768,0.012895
3,A1-none-4-1,A/Alagoas/4131/2025_H1N1,4,none,A1,1704,4657768,0.000183
4,A1-none-4-1,A/Alaska/63/2025_H3N2,4,none,A1,34467,4657768,0.003700
...,...,...,...,...,...,...,...,...
1819,H2-none-512-2,A/Victoria/1493/2025_H1N1,512,none,H2,652,2451786,0.000133
1820,H2-none-512-2,A/Victoria/4897/2022_IVR-238_H1N1,512,none,H2,4296,2451786,0.000876
1821,H2-none-512-2,A/Wisconsin/588/2019_H1N1,512,none,H2,1004,2451786,0.000205
1822,H2-none-512-2,A/Wisconsin/67/2022_H1N1,512,none,H2,3208,2451786,0.000654


We now have this fraction of reads devoted to all strains calculated for all wells. However, ideally we should just focus on those wells containing dilutions that we would use for actual neutralization assays. We should choose a set of replicate wells where the fraction of neutralization standard reads begins to increase linearly with the increasing reciprocal dilution factor. See plots above for choosing these wells. 

In [12]:
# Choose wells
well1 = 'E1'
well2 = 'E2'

In [13]:
# Choose a pair of replicate wells near the beginning of the linear range
single_well = merged_df.loc[merged_df['sample'].str.contains(f'{well1}-|{well2}-')]

In [14]:
# Calculate mean fraction strain across both wells
mean_df = single_well.groupby(['strain'])['fraction_strain'].mean().to_frame().rename(columns = {'fraction_strain': 'mean_fraction_strains'}).reset_index()
mean_single_well = single_well.merge(mean_df, on = 'strain', how = 'left')

# calcualte ratios to add for equal pool
num_strains = len(mean_single_well.strain.unique())
mean_single_well['ratio_to_add'] = (1/num_strains)/mean_single_well['fraction_strain']
mean_single_well['mean_ratio_to_add'] = (1/num_strains)/mean_single_well['mean_fraction_strains']

mean_single_well['est_tcid50'] = (mean_single_well['mean_fraction_strains']*25000)*76

print(f'this library has {num_strains} total strains')
print('stats where there isnt 0 of a virus...')
print(mean_single_well.query('mean_ratio_to_add != inf')[['mean_ratio_to_add']].describe())

print('\nviruses with 0 titer...')
print(mean_single_well.query('mean_ratio_to_add == inf').strain.unique())

ratio_cutoff = 250
print(f'\nviruses with >0 titer but ratio >={ratio_cutoff} to increase...')
print(mean_single_well.query('mean_ratio_to_add != inf').query(f'mean_ratio_to_add >= {ratio_cutoff}').strain.unique())

this library has 114 total strains
stats where there isnt 0 of a virus...
       mean_ratio_to_add
count         210.000000
mean          175.325823
std          1691.329030
min             0.275457
25%             1.416315
50%             3.801838
75%            16.469363
max         17381.936373

viruses with 0 titer...
['A/Aberystwyth/5654/2025_H3N2' 'A/Bahrain/4684/2025_H3N2'
 'A/BritishColumbia/PHL-2467/2025_H1N1'
 'A/Catalonia/NSAV198331894/2025_H1N1' 'A/Delaware/81/2025_H3N2'
 'A/England/1845724/2025_H3N2' 'A/Galicia/GA-CHUAC-449/2025_H1N1'
 'A/SouthAustralia/2523011822/2025_H1N1' 'A/SuratThani/P3371/2025_H3N2']

viruses with >0 titer but ratio >=250 to increase...
['A/Bangkok/P2277/2025_H1N1']


## Re-pooling calculations

In [15]:
# Factor to multiply ratios by
repool_factor = 15

In [20]:
# Get library IDs
lib_id_df=pd.read_csv('../../library_design/construct_order/flu-seqneut-2025to2026-barcode-to-strain-designed.csv')

# Get old strain metadata
h3_metadata_df=pd.read_csv('../../library_design/initial_design/data/2025-NH-VCM-neutralization-library-strain-selection-H3N2.tsv', sep='\t')
h1_metadata_df=pd.read_csv('../../library_design/initial_design/data/2025-NH-VCM-neutralization-library-strain-selection-H1N1.tsv', sep='\t')

h3_metadata_df['subtype'] = 'H3N2'
h1_metadata_df['subtype'] = 'H1N1'
h3_metadata_df['strain'] = h3_metadata_df['representative_strain'] + '_' + h3_metadata_df['subtype']
h1_metadata_df['strain'] = h1_metadata_df['representative_strain'] + '_' + h1_metadata_df['subtype']

metadata_df = pd.concat([h3_metadata_df, h1_metadata_df])
metadata_df = metadata_df.rename(columns={
    'derived_haplotype': 'old_derived_haplotype',
    'count': 'old_count',
    'latest_sequence': 'old_latest_sequence',
})

# Get new strain metadata
new_h3_metadata_df=pd.read_csv('../data/h3n2_haplotypes_updated_from_John_2026-01-09.tsv', sep='\t')
new_h1_metadata_df=pd.read_csv('../data/h1n1pdm_haplotypes_updated_from_John_2026-01-09.tsv', sep='\t')
new_metadata_df = pd.concat([new_h3_metadata_df, new_h1_metadata_df])

# # Merge metadata on HA sequence -- keep old representative strains
metadata_df = metadata_df.merge(new_metadata_df, on='representative_strain_ha_sequence', how='left')

# Make repool dataframe using chosen well 
repool_df = (mean_single_well
             .query('mean_ratio_to_add != inf')
             .query(f'well == "{well2}"')
             [['strain','fraction_strain','mean_ratio_to_add']]
             .assign(x_volume_to_add = lambda x: x['mean_ratio_to_add'] * repool_factor)
             .merge(lib_id_df, how='outer')
             .assign(
                 subtype = lambda x: x['name'].str.replace('flu-seqneut-25to26_','').str.split('_').str[0],
                 number = lambda x: pd.to_numeric(
                     x['name'].str.replace('flu-seqneut-25to26_','').str.split('_').str[1],  # removed inner lambda
                     errors='coerce').fillna(1e6).astype(int),  # use a big number to push NaNs to bottom
                 strain_id = lambda x: x['subtype'] + '_' + x['number'].astype(str)
             )
             .sort_values(by=['subtype', 'number'], ascending=True)
             .drop(columns=['number', 'name', 'barcode', 'nt_sequence_HA_ectodomain','protein_sequence_HA_ectodomain'])  # number is now temporary
             .drop_duplicates()
             .dropna(subset=['fraction_strain'])
             .reset_index(drop=True)
)

pooling_mathdir = '../results/pooling_math'
os.makedirs(pooling_mathdir, exist_ok=True)
repool_df.to_csv(os.path.join(pooling_mathdir, '2025-07-16_repooling_math.csv'), index=False)

# drop strains
strains_to_drop = [
    'A/Bangkok/P2277/2025_H1N1', # potential extremely low level plasmid contamination
    # 'A/Chinautla/FLU-331/2025_H1N1', # low titer
    # 'A/SouthAustralia/2518902563/2025_H1N1' # low titer
]
repool_df = repool_df[~repool_df['strain'].isin(strains_to_drop)]

# Merge with metadata
repool_df = repool_df.merge(metadata_df[['old_derived_haplotype', 'derived_haplotype','strain','count','latest_sequence', 'representative_strain_ha_sequence']], on='strain', how='left')

# print('Displaying the top 30 lowest titer strains...')
repool_df.sort_values(by='x_volume_to_add', ascending=False).head(30).reset_index(drop=True)


# repool_df.to_csv('../results/flu-seqneut-2025_viral_library_old_metadata.csv',index=False)

,strain,fraction_strain,mean_ratio_to_add,x_volume_to_add,accession,strain_type,subtype,subclade,num_date,vaccine_type,strain_id,old_derived_haplotype,derived_haplotype,count,latest_sequence,representative_strain_ha_sequence
0,A/Chinautla/FLU-331/2025_H1N1,0.000113,81.921046,1228.815689,PV500381_A158T_V169I,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_44,"D.3.1:S69P,A141T,V152I","D.3.1:S69P,A141T,V152I",3.0,2025-08-16,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
1,A/SouthAustralia/2518902563/2025_H1N1,0.000197,57.962709,869.440632,PV728578_F12S_P288Q_V338I,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_46,"D.3.1:I267T,P271Q,V321I",NaN,NaN,NaN,MKAILVVMLYTSTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
2,A/Victoria/1493/2025_H1N1,0.000211,47.345466,710.181985,PV617059_Q180K,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_13,D.3.1:Q163K,D.3.1:Q163K,2.0,2025-08-08,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
3,A/Queensland/159/2025_H1N1,0.000195,39.457442,591.861635,PX400159_S207R,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_19,D.3.1:S190R,D.3.1:S190R,7.0,2025-09-09,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
4,A/BritishColumbia/PHL-2478/2025_H1N1,0.000258,32.062627,480.939404,PV585284_E83K_S551G,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_33,"D.3.1:E66K,S157L","D.3.1:E66K,S157L",2.0,2025-08-14,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
5,A/SouthAustralia/2518911830/2025_H1N1,0.000145,30.820402,462.306026,PV896643_Q71L_A278S,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_35,"D.3.1:K43R,Q54L,A261S","D.3.1:K43R,Q54L,A261S",4.0,2025-08-06,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
6,A/Lao/820-2765/2025_H1N1,0.000269,30.454001,456.810015,PX445514_E277K,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_14,D.3.1:E260K,D.3.1:E260K,9.0,2025-11-23,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
7,A/Alagoas/4131/2025_H1N1,0.000139,28.787229,431.808431,PX400159_A278S,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_7,D.3.1:A261S,NaN,NaN,NaN,MKAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
8,A/NewSouthWales/2525913630/2025_H1N1,0.000339,27.424606,411.369089,PV733162_K2E_I202T,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_39,"D.3.1:D86N,I185T","D.3.1:D86N,I185T",4.0,2025-10-03,MEAILVVMLYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
9,A/Catalonia/NSVH102686666/2025_H1N1,0.000328,26.255198,393.827972,PV885931_M8I_N490D,circulating_2025to2026,H1N1,NaN,NaN,NaN,H1N1_42,D.3.1:V152I,NaN,NaN,NaN,MKAILVVILYTFTTANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...


In [22]:
repool_df.query('subtype=="H3N2"')

,strain,fraction_strain,mean_ratio_to_add,x_volume_to_add,accession,strain_type,subtype,subclade,num_date,vaccine_type,strain_id,old_derived_haplotype,derived_haplotype,count,latest_sequence,representative_strain_ha_sequence
44,A/Norway/8765/2025_H3N2,0.005271,1.734784,26.021756,PX422923,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_1,J.2.4.1,NaN,NaN,NaN,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
45,A/NewSouthWales/2526015582/2025_H3N2,0.007344,1.199615,17.994231,PX422923_V104I,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_2,J.2.4.1:V88I,K:V88I,93.0,2025-12-15,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
46,A/Bangkok/P2390/2025_H3N2,0.005057,1.851857,27.777858,PX422972,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_3,J.2.4.1:A272T,NaN,NaN,NaN,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
47,A/SouthAustralia/2527003588/2025_H3N2,0.005473,1.416315,21.244729,PX422923_K223Q_V239I,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_4,"J.2.4.1:K207Q,V223I","K:K207Q,V223I",16.0,2025-11-14,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
48,A/Queensland/2526516675/2025_H3N2,0.002599,3.104287,46.564309,PX422923_V104I_I230T,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_5,"J.2.4.1:V88I,I214T",NaN,NaN,NaN,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
49,A/SouthAustralia/2527902708/2025_H3N2,0.005289,1.604479,24.067187,PX422923_S112C_K223Q_V239I,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_6,"J.2.4.1:S96C,K207Q,V223I","K:S96C,K207Q,V223I",4.0,2025-10-30,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
50,A/Bangkok/P2572/2025_H3N2,0.006210,1.500743,22.511147,PX422972_S295T,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_7,"J.2.4.1:A272T,S279T","K:A272T,S279T",1.0,2025-08-14,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
51,A/DistritoFederal/250106000728/2025_H3N2,0.004076,2.313874,34.708104,PV721449,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_8,J.2.3:S145N,J.2.3:S145N,48.0,2025-11-13,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...
52,A/Malaysia/IMR/SARI/2517/2025_H3N2,0.004949,1.798555,26.978326,PX422923_F95V,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_9,J.2.4.1:F79V,K:F79V,2.0,2025-11-10,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...
53,A/Norway/8976/2025_H3N2,0.005317,1.511090,22.666344,PX422923_I83M,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_10,J.2.4.1:I67M,K:I67M,7.0,2025-11-19,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...


In [17]:
print(f'Adding {repool_factor}x of each strain ratio...')
_sum = (
    sum(repool_df.mean_ratio_to_add) * repool_factor
     )
print(_sum, 'uL total pool')
print('This means adding strains in volumes ranging from...')
print(repool_df.x_volume_to_add.min(), 'uL to ', repool_df.x_volume_to_add.max(), 'uL')
print('Assuming worse case scenario of 1:16 on 150k cells...')
volume_per_plate = (50/16)*110
print(volume_per_plate, 'uL per plate')
print('About how many plates can I run?')
print((_sum-10)/volume_per_plate)

Adding 15x of each strain ratio...
15567.431641408482 uL total pool
This means adding strains in volumes ranging from...
4.131862036549576 uL to  1228.8156888622905 uL
Assuming worse case scenario of 1:16 on 150k cells...
343.75 uL per plate
About how many plates can I run?
45.25798295682468


## Visualize barcode- and strain-level balancing in the current pool

In [18]:
# Plot the current fraction of each strain in the pool
strains_chart = (
    alt.Chart(mean_single_well)
    .encode(
        alt.X(
            "fraction_strain",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("strain"),
        
        tooltip = ['strain', 'fraction_strain', 'est_tcid50'],
    )
).mark_bar(height={"band": 0.85}).properties(
        height=alt.Step(10),
        width=250,
        title="",
    ).properties(
        height = alt.Step(10),
        width = 200,
        title = "Strain representation, initial pool")

# add veritcal line where we would expect equal representation of all barcodes in pool
expected_line = alt.Chart(
    pd.DataFrame({'x': [1/num_strains]})
).mark_rule(strokeDash = [2,2], strokeWidth = 2).encode(x = 'x')

# plot both barcode counts and expected line
strains_chart + expected_line

alt.LayerChart(...)

In [19]:
# Each barcode fraction across strains
all_barcode_counts = counts[['strain', 'barcode', 'count', 'well']].dropna()
single_well_all_barcode_counts = all_barcode_counts[all_barcode_counts['well'].isin([f'{well1}',f'{well2}'])]

# Get tidy single well means
tidy_single_well = single_well_all_barcode_counts[['strain','barcode','count']].groupby(['strain', 'barcode']).mean().reset_index()
# Get sums for each strain
strain_sums_df = tidy_single_well.groupby('strain').sum().rename(columns = {'count': 'strain_count_sum'}).reset_index()
# Merge and calculate per strain the fraction represented by each barcode
tidy_single_well = tidy_single_well.merge(strain_sums_df[['strain', 'strain_count_sum']], 
                       on = ['strain'],
                       validate="many_to_one",
                      )
tidy_single_well['per_strain_fraction_barcode'] = tidy_single_well['count'] / tidy_single_well['strain_count_sum']
tidy_single_well['barcode_letter'] = tidy_single_well.groupby('strain').cumcount().apply(lambda x: string.ascii_uppercase[x])

# Plot as colored bar chart
bar_chart = alt.Chart(tidy_single_well).mark_bar(height={"band": 0.85}).encode(
    x = 'per_strain_fraction_barcode',
    y = 'strain',
    color=alt.Color('barcode_letter', legend=None).scale(range=color_palette),
    tooltip = ['strain', 'per_strain_fraction_barcode', 'barcode'],
).configure_axis(grid=False).properties(
        height = alt.Step(10),
        width = 200,
        title = "Barcode fraction for each strain, initial pool")

bar_chart

alt.Chart(...)